# Predicting Airline Satisfaction: Exploratory Data Analysis

Kaggle Playground Series S6E10 (`playground-series-s6e10`). Binary
classification: predict the probability that `satisfaction` is true for
each test `id`, scored by **ROC AUC** (Kaggle API, 2026-10-02).

This notebook is the executable source of truth for the EDA. It uses a
lightweight CRISP-DM structure: this run concentrates on business and data
understanding, then records implications for preparation, modeling,
evaluation, and eventual submission. It is authored locally and executed
on Kaggle. CPU only; internet disabled.


## 1. Business Understanding

### 1.1 Objective and success measure

Predict a probability of passenger satisfaction for every test row. The
competition ranks submissions by ROC AUC, so discrimination and stable
out-of-fold validation matter; a classification threshold does not.

### 1.2 Scope and execution contract

This is a Playground competition: prefer a small number of validated
tabular models over broad experimentation. Data understanding happens here;
model selection and submission remain separate, evidence-backed stages.

## 2. Data Understanding

### 2.1 Environment


In [ ]:
import json
import platform
import shutil
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from scipy.stats import ks_2samp
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import OrdinalEncoder

import seaborn as sns

sns.set_theme(style="whitegrid")
PALETTE = "viridis"
print("python", platform.python_version())
print({
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
    "matplotlib": matplotlib.__version__,
})


### 2.2 Configuration and data loading


In [ ]:
SEED = 42
NOTEBOOK_VERSION = "v2"
TARGET = "satisfaction"
ID_COLUMN = "id"
ADV_SAMPLE_SIZE = 200_000

SURVEY_FEATURES = [
    "Inflight wifi service",
    "Departure/Arrival time convenient",
    "Ease of Online booking",
    "Gate location",
    "Food and drink",
    "Online boarding",
    "Seat comfort",
    "Inflight entertainment",
    "On-board service",
    "Leg room service",
    "Baggage handling",
    "Checkin service",
    "Cleanliness",
]
NUMERIC_FEATURES = [
    "Age",
    "Flight Distance",
    "Departure Delay in Minutes",
    "Arrival Delay in Minutes",
    *SURVEY_FEATURES,
]
CATEGORICAL_FEATURES = [
    "Gender",
    "Customer Type",
    "Type of Travel",
    "Class",
]
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES


def find_data_dir() -> Path:
    """Locate competition files on Kaggle or in a local data folder.

    The mount path is not assumed. Known layouts are tried first, then
    the input tree is walked (master standard section 12).

    Returns:
        Directory that contains train.csv.

    Raises:
        FileNotFoundError: If train.csv is not on any tried path.
    """
    candidates = [
        Path("/kaggle/input/competitions/playground-series-s6e10"),
        Path("/kaggle/input/playground-series-s6e10"),
        Path("../data"),
        Path("data"),
    ]
    for cand in candidates:
        if (cand / "train.csv").exists():
            return cand
    mount = Path("/kaggle/input")
    if mount.exists():
        hits = sorted(mount.rglob("train.csv"))
        if hits:
            return hits[0].parent
    raise FileNotFoundError("train.csv not found in any known location")


DATA_DIR = find_data_dir()
print(f"DATA_DIR = {DATA_DIR.resolve()}")


In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")

assert train.shape == (699_635, 23), train.shape
assert test.shape == (299_844, 22), test.shape
assert list(sample.columns) == [ID_COLUMN, TARGET]
assert sorted(ALL_FEATURES) == sorted(
    c for c in train.columns if c not in (ID_COLUMN, TARGET)
)
y = train[TARGET].astype(int)
assert set(y.unique()) <= {0, 1}
print("train", train.shape, "test", test.shape, "positive", int(y.sum()))
train.dtypes.to_frame("dtype")


### 2.3 Target and baseline

The sample submission is a constant probability. The check below is
whether that constant matches the train positive rate within floating-point
precision, not whether the two floats are bit-identical.


In [ ]:
positive = int(y.sum())
rate = positive / len(y)
sample_constant = float(sample[TARGET].iloc[0])
print(
    f"positive {positive} / {len(y)} = {rate:.16f}\n"
    f"sample constant {sample_constant:.16f}\n"
    f"absolute difference {abs(rate - sample_constant):.3e}\n"
    f"sample is constant: {sample[TARGET].nunique() == 1}"
)


### 2.4 Data quality: missing arrival delay

`Arrival Delay in Minutes` is the only nullable column in the local
read. This section checks that is still true on the mounted files, and
whether a missing arrival delay has a different satisfaction rate.


In [ ]:
train_nulls = train.isna().sum()
test_nulls = test.isna().sum()
print("train nulls")
print(train_nulls[train_nulls > 0].to_string())
print("test nulls")
print(test_nulls[test_nulls > 0].to_string())

arrival = "Arrival Delay in Minutes"
missing = train[arrival].isna()
print(
    f"null rate train {missing.mean():.6f} "
    f"({int(missing.sum())}), test {test[arrival].isna().mean():.6f}"
)
by_missing = y.groupby(missing).agg(["mean", "size"])
by_missing.index = by_missing.index.map({False: "observed", True: "missing"})
print(by_missing.to_string())
observed = train.loc[~missing, ["Departure Delay in Minutes", arrival]]
print(
    "corr(departure, arrival | observed) = "
    f"{observed.corr().iloc[0, 1]:.6f}"
)


### 2.5 Survey scales and score 0

Most survey columns include 0. `Baggage handling` starts at 1. The
question is whether 0 behaves like the bottom of a 0–5 scale or like a
separate "not applicable" bucket.


In [ ]:
rows = []
for col in SURVEY_FEATURES:
    values = train[col]
    zero = values == 0
    rows.append({
        "feature": col,
        "zero_share": float(zero.mean()),
        "rate_at_0": float(y[zero].mean()) if zero.any() else np.nan,
        "rate_at_1_to_5": float(y[~zero].mean()),
        "min": int(values.min()),
        "max": int(values.max()),
    })
zero_table = pd.DataFrame(rows).sort_values("zero_share", ascending=False)
print(zero_table.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

fig, ax = plt.subplots(figsize=(8, 5))
plot_df = zero_table.dropna(subset=["rate_at_0"])
ax.scatter(
    plot_df["rate_at_1_to_5"],
    plot_df["rate_at_0"],
    c=np.linspace(0, 1, len(plot_df)),
    cmap=PALETTE,
    s=40,
)
limits = [0, 1]
ax.plot(limits, limits, color="0.5", lw=1)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("Satisfaction rate at scores 1-5")
ax.set_ylabel("Satisfaction rate at score 0")
ax.set_title("Does survey score 0 match the rest of the scale?")
fig.tight_layout()


### 2.6 Categorical target rates


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, col in zip(axes.ravel(), CATEGORICAL_FEATURES):
    rates = y.groupby(train[col]).mean().sort_values()
    colors = sns.color_palette(PALETTE, n_colors=len(rates))
    rates.plot.barh(ax=ax, color=colors)
    ax.set_xlim(0, 1)
    ax.set_xlabel("Satisfaction rate")
    ax.set_title(col)
fig.tight_layout()

for col in CATEGORICAL_FEATURES:
    rates = y.groupby(train[col]).agg(["mean", "size"]).sort_values("mean")
    print(f"\n{col}")
    print(rates.to_string(float_format=lambda v: f"{v:.4f}"))


### 2.7 Univariate signal


In [ ]:
def univariate_auc(values: pd.Series, labels: pd.Series) -> float:
    """ROC AUC of one numeric column, ignoring missing values.

    Args:
        values: Feature values aligned with labels.
        labels: Binary target.

    Returns:
        AUC, folded so the reported number is at least 0.5.
    """
    mask = values.notna()
    score = roc_auc_score(labels[mask], values[mask])
    return float(max(score, 1.0 - score))


auc_rows = []
for col in NUMERIC_FEATURES:
    auc_rows.append({
        "feature": col,
        "auc": univariate_auc(train[col], y),
        "missing": int(train[col].isna().sum()),
    })
auc_table = pd.DataFrame(auc_rows).sort_values("auc", ascending=False)
print(auc_table.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

fig, ax = plt.subplots(figsize=(8, 5))
colors = sns.color_palette(PALETTE, n_colors=len(auc_table))
ax.barh(auc_table["feature"], auc_table["auc"], color=colors)
ax.set_xlim(0.5, max(0.75, float(auc_table["auc"].max()) + 0.02))
ax.set_xlabel("Univariate ROC AUC (direction folded)")
ax.set_title("Single-column signal")
fig.tight_layout()


### 2.8 Train-versus-test drift

Numeric columns use a two-sample KS statistic. Categorical columns use
the largest absolute difference in level share.


In [ ]:
drift_rows = []
for col in NUMERIC_FEATURES:
    left = train[col].dropna()
    right = test[col].dropna()
    stat = ks_2samp(left, right).statistic
    drift_rows.append({
        "feature": col,
        "kind": "numeric",
        "stat": float(stat),
    })
for col in CATEGORICAL_FEATURES:
    train_share = train[col].value_counts(normalize=True)
    test_share = test[col].value_counts(normalize=True)
    aligned = pd.concat([train_share, test_share], axis=1).fillna(0.0)
    delta = (aligned.iloc[:, 0] - aligned.iloc[:, 1]).abs().max()
    drift_rows.append({
        "feature": col,
        "kind": "categorical",
        "stat": float(delta),
    })
drift = pd.DataFrame(drift_rows).sort_values("stat", ascending=False)
print(drift.to_string(index=False, float_format=lambda v: f"{v:.6f}"))
print("max stat", float(drift["stat"].max()))


### 2.9 Adversarial validation

A classifier tries to tell train from test. This uses a stratified
subsample of `ADV_SAMPLE_SIZE` rows from each side, seed 42, so the
number is a screen rather than a full-data measurement.


In [ ]:
rng = np.random.default_rng(SEED)
train_idx = rng.choice(len(train), size=ADV_SAMPLE_SIZE, replace=False)
test_idx = rng.choice(len(test), size=ADV_SAMPLE_SIZE, replace=False)
adv_train = train.iloc[train_idx][ALL_FEATURES].copy()
adv_test = test.iloc[test_idx][ALL_FEATURES].copy()
adv = pd.concat([adv_train, adv_test], axis=0)
adv_y = np.r_[
    np.zeros(len(adv_train), dtype=int),
    np.ones(len(adv_test), dtype=int),
]
encoder = OrdinalEncoder(encoded_missing_value=-1)
adv[CATEGORICAL_FEATURES] = encoder.fit_transform(adv[CATEGORICAL_FEATURES])
adv = adv.fillna(-1)
model = HistGradientBoostingClassifier(
    max_iter=80,
    learning_rate=0.1,
    random_state=SEED,
)
folds = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
adv_pred = cross_val_predict(
    model, adv, adv_y, cv=folds, method="predict_proba", n_jobs=1,
)[:, 1]
adv_auc = float(roc_auc_score(adv_y, adv_pred))
print(
    f"adversarial OOF AUC {adv_auc:.4f} "
    f"on {ADV_SAMPLE_SIZE} rows per side, 3-fold"
)


## 3. Data Preparation Implications

- Exclude `id` from predictors; it is an alignment key, not an observed
  passenger attribute.
- Preserve survey score 0 as an observed value. Its meaning varies by
  column, so do not globally convert 0 to missing or impose one monotone
  interpretation.
- Handle the rare missing arrival delay inside each validation fold. Pair a
  simple imputation with a missingness indicator so the model can retain the
  small rate difference without leakage.
- Use one train/test feature recipe. The drift screen gives no reason for
  dataset-specific correction or reweighting.

### 3.1 Reproducibility summary


In [ ]:
print("NOTEBOOK_VERSION", NOTEBOOK_VERSION)
print("rows", len(train), len(test))
print("positive_rate", f"{y.mean():.10f}")
print("arrival_null_train", int(train[arrival].isna().sum()))
print("top_univariate", auc_table.iloc[0].to_dict())
print("max_drift", float(drift["stat"].max()))
print("adversarial_auc", round(adv_auc, 4))
print("duplicate_feature_rows", int(
    train.drop(columns=[ID_COLUMN, TARGET]).duplicated().sum()
))


## 4. Modeling Implications

- Start with CatBoost as the primary baseline because it handles the four
  categorical columns, nonlinear survey scores, and missing values directly.
- Use one strong gradient-boosting challenger, such as LightGBM, to test
  whether a different tree implementation adds meaningful OOF gain or useful
  prediction diversity. Avoid a broad model zoo.
- Reuse one fixed stratified fold definition across candidates. Any learned
  imputation, encoding, calibration, or target-derived transform must fit
  inside the fold.
- Treat `Class`, `Type of Travel`, and `Online boarding` as strong baseline
  signals, not causal conclusions. Add interactions only after the baseline
  shows a reproducible gap worth pursuing.


## 5. Evaluation and Next Steps

### 5.1 Evaluation contract

The modeling notebook should report overall and per-fold ROC AUC, retain
aligned OOF predictions, and compare challengers against the standing model
with a predeclared paired promotion rule. The 0.5004 adversarial AUC is a
subsample screen, not proof that every hidden-test distribution is identical.
Target-rate tables are descriptive and must not be read causally.

### 5.2 Next step and deployment boundary

Record the fold definition and baseline design in `docs/3_implementation_plan.md`
before the first model run. In CRISP-DM terms, deployment here means a
verified `submission.csv` produced by a completed Kaggle kernel and recorded
in `docs/5_submission_manifest.md`; it is intentionally outside this EDA run.


## Appendix: Self-export

Kaggle does not return an executed notebook through its API. This cell
copies `/kaggle/working/__notebook__.ipynb`, which holds outputs of the
cells above. It stays last: the copy does not include this cell's output.


In [ ]:
source = Path("/kaggle/working/__notebook__.ipynb")
if source.exists():
    dest = Path("/kaggle/working/executed_notebook.ipynb")
    shutil.copy(source, dest)
    exported = json.loads(dest.read_text())
    with_outputs = sum(1 for cell in exported["cells"] if cell.get("outputs"))
    print(
        f"self-export: {dest.name} — {len(exported['cells'])} cells, "
        f"{with_outputs} with outputs, {dest.stat().st_size / 1024:.0f} KB"
    )
else:
    print("no __notebook__.ipynb — self-export skipped")
